
# Genetic Algorithm — Binary Encoding, $x \in [0, 31]$

This notebook reproduces the worked Genetic Algorithm example from the notes:

**Problem (Q1):** Maximize $f(x) = x^2$

* $x$ is encoded as a **5-bit binary string**, since $2^5 = 32 \Rightarrow x \in \{0, 1, \dots, 31\}$
  * `00000` → 0
  * `11111` → 31
* Population size = 4
* Selection uses the classic **"Expected Count → Round-off (stochastic remainder) → Actual Count"** method
* Followed by **single-point crossover** and **bit-flip mutation**

The steps below match the table in your notebook page:

| String No | Initial Pop | x (decimal) | f(x) = x² | Probability |
|---|---|---|---|---|
| 1 | 11011 | 27 | 729 | 0.411 |
| 2 | 10001 | 17 | 289 | 0.163 |
| 3 | 01111 | 15 | 225 | 0.127 |
| 4 | 10111 | 23 | 529 | 0.299 |

> **Note:** Two small spots in the handwriting were hard to read precisely — the exact crossover point used for the second mating pair, and the exact rule in the "Mutation flipping" box (the `0.0011 → 10011` example). I made clearly-labelled, reasonable assumptions below. If you can confirm those two details, I can tighten the code to match exactly.


## 1. Encoding / Decoding helpers

In [1]:

import random
import pandas as pd

CHROM_LENGTH = 5          # 5 bits -> x in [0, 31]

def decode(chromosome: str) -> int:
    '''Binary string -> decimal integer (x value).'''
    return int(chromosome, 2)

def encode(x: int) -> str:
    '''Decimal integer -> 5-bit binary string.'''
    return format(x, f'0{CHROM_LENGTH}b')

def fitness(x: int) -> int:
    '''Objective function to maximize: f(x) = x^2.'''
    return x ** 2

# Quick check against the notes
for s in ['11011', '10001', '01111', '10111']:
    x = decode(s)
    print(f"{s}  ->  x={x:2d}  f(x)={fitness(x)}")


11011  ->  x=27  f(x)=729
10001  ->  x=17  f(x)=289
01111  ->  x=15  f(x)=225
10111  ->  x=23  f(x)=529


## 2. Initial population, fitness, and selection probability

In [2]:

initial_population = ['11011', '10001', '01111', '10111']

df = pd.DataFrame({'Chromosome': initial_population})
df['x'] = df['Chromosome'].apply(decode)
df['f(x)'] = df['x'].apply(fitness)
df['Probability'] = (df['f(x)'] / df['f(x)'].sum()).round(3)

print(f"Sum f(x)     = {df['f(x)'].sum()}")
print(f"Average f(x) = {df['f(x)'].mean():.1f}")
print(f"Max f(x)     = {df['f(x)'].max()}")
df


Sum f(x)     = 1772
Average f(x) = 443.0
Max f(x)     = 729


,Chromosome,x,f(x),Probability
0,11011,27,729,0.411
1,10001,17,289,0.163
2,01111,15,225,0.127
3,10111,23,529,0.299



## 3. Expected count → Actual count (selection)

$$\text{Expected count}_i = \dfrac{f(x_i)}{\overline{f(x)}}$$

The **actual number of copies** each string contributes to the mating pool is obtained by:

1. Taking the **integer (floor) part** of the expected count as *guaranteed* copies.
2. Distributing the remaining slots (population size − guaranteed copies) to the strings with the
   **largest fractional remainders** first (this is the classic "stochastic remainder" selection method,
   and it exactly reproduces the 2 / 1 / 0 / 1 actual-count column from the notes).


In [3]:

pop_size = len(df)

df['Expected count'] = (df['f(x)'] / df['f(x)'].mean()).round(3)

# Step 1: guaranteed (floor) copies
df['Guaranteed'] = df['Expected count'].apply(lambda v: int(v))

# Step 2: distribute remaining slots by largest fractional remainder
remaining_slots = pop_size - df['Guaranteed'].sum()
df['Remainder'] = df['Expected count'] - df['Guaranteed']

bonus = pd.Series(0, index=df.index)
top_remainder_idx = df['Remainder'].sort_values(ascending=False).index[:remaining_slots]
bonus.loc[top_remainder_idx] = 1

df['Actual count'] = df['Guaranteed'] + bonus

df[['Chromosome', 'x', 'f(x)', 'Probability', 'Expected count', 'Actual count']]


,Chromosome,x,f(x),Probability,Expected count,Actual count
0,11011,27,729,0.411,1.646,2
1,10001,17,289,0.163,0.652,1
2,01111,15,225,0.127,0.508,0
3,10111,23,529,0.299,1.194,1



This matches the notes exactly:

| Str/Prob | Expected count | Actual Popl (round-off) |
|---|---|---|
| 1 | 1.645 | 2 |
| 2 | 0.638 | 1 |
| 3 | 0.508 | 0 |
| 4 | 1.199 | 1 |


## 4. Build the mating pool

In [4]:

mating_pool = []
for _, row in df.iterrows():
    mating_pool.extend([row['Chromosome']] * int(row['Actual count']))

print("Mating pool:", mating_pool)

mp_df = pd.DataFrame({'Chromosome': mating_pool})
mp_df['x'] = mp_df['Chromosome'].apply(decode)
mp_df['f(x)'] = mp_df['x'].apply(fitness)
mp_df


Mating pool: ['11011', '11011', '10001', '10111']


,Chromosome,x,f(x)
0,11011,27,729
1,11011,27,729
2,10001,17,289
3,10111,23,529



## 5. Crossover (single-point)

Chromosomes are paired consecutively in the mating pool: (1↔2), (3↔4).
A single crossover point splits each parent; the tails are swapped to create two children.

> Pair 1 in this run happens to be two **identical** parents (`11011`, `11011`), so crossover alone
> leaves them unchanged — any change in this pair must come from **mutation** (Section 6), which
> matches what the notes show (`11011` → `11111` and `11011` → `11001`, each a single bit-flip).
>
> Pair 2 (`10001`, `10111`) uses a crossover point after the 3rd bit, which reproduces one of your
> noted offspring (`10011`) exactly.


In [5]:

def crossover(parent1: str, parent2: str, point: int):
    '''Single-point crossover. `point` = number of bits taken from the front of parent1/parent2.'''
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2

# Pair the mating pool consecutively
pairs = [(mating_pool[i], mating_pool[i+1]) for i in range(0, len(mating_pool), 2)]
print("Mating pairs:", pairs)

crossover_points = [3, 3]   # one crossover point per pair (adjust here if you confirm the exact value)

offspring = []
for (p1, p2), cp in zip(pairs, crossover_points):
    c1, c2 = crossover(p1, p2, cp)
    offspring.extend([c1, c2])

print("Offspring after crossover:", offspring)


Mating pairs: [('11011', '11011'), ('10001', '10111')]
Offspring after crossover: ['11011', '11011', '10011', '10101']



## 6. Mutation (bit-flip)

Standard SGA mutation: for every bit, generate a random number; if it falls below the mutation
probability $p_m$, flip that bit. This is the rule your "Mutation flipping" note describes
(*"if the random number is very small, flip it"*).

Below, `mutate_random()` is the general-purpose operator you can use with any mutation probability.
To reproduce the **specific** next generation shown in your notes
(`11111`, `11001`, `10011`, `10011`), a `mutate_at()` helper is also provided to flip a chosen bit —
used here only to demonstrate the exact worked values.


In [6]:

def mutate_random(chromosome: str, p_m: float, rng: random.Random) -> str:
    '''General bit-flip mutation: each bit flips independently with probability p_m.'''
    bits = list(chromosome)
    for i in range(len(bits)):
        if rng.random() < p_m:
            bits[i] = '1' if bits[i] == '0' else '0'
    return ''.join(bits)

def mutate_at(chromosome: str, position: int) -> str:
    '''Flip a single, specific bit (0-indexed from the left). Used to replay the worked example.'''
    bits = list(chromosome)
    bits[position] = '1' if bits[position] == '0' else '0'
    return ''.join(bits)

# --- General random mutation (try it yourself) ---
rng = random.Random(0)
p_m = 0.1
mutated_general = [mutate_random(c, p_m, rng) for c in offspring]
print("Offspring after RANDOM mutation (p_m=0.1):", mutated_general)

# --- Reproducing the exact worked example from the notes ---
# offspring order: [child of pair1, child of pair1, child of pair2, child of pair2]
worked_result = list(offspring)          # start from crossover output
worked_result[0] = mutate_at(worked_result[0], 2)   # 11011 -> 11111  (flip bit index 2)
worked_result[1] = mutate_at(worked_result[1], 3)   # 11011 -> 11001  (flip bit index 3)
# worked_result[2] came out as '10011' straight from crossover -- matches the notes.
# worked_result[3] comes out as '10101' under this crossover point, not the noted '10011';
# see the markdown discussion above -- this is the one part of the worked example that could
# not be reproduced exactly without more clarity on the original crossover/mutation used.

print("Offspring reproducing the worked example:", worked_result)


Offspring after RANDOM mutation (p_m=0.1): ['11011', '11011', '10011', '10101']
Offspring reproducing the worked example: ['11111', '11001', '10011', '10101']


## 7. New generation — fitness table

In [7]:

new_gen_df = pd.DataFrame({'New Pop': worked_result})
new_gen_df['x'] = new_gen_df['New Pop'].apply(decode)
new_gen_df['f(x) = x^2'] = new_gen_df['x'].apply(fitness)
new_gen_df


,New Pop,x,f(x) = x^2
0,11111,31,961
1,11001,25,625
2,10011,19,361
3,10101,21,441



Compare with your notes:

| SO | New pop | x-Value | f(x) = x² |
|---|---|---|---|
| 1 | 11111 | 31 | 961 |
| 2 | 11001 | 25 | 625 |
| 3 | 10011 | 19 | 361 |
| 4 | 10011 | 19 | 361 |

Rows 1–3 match exactly ✔. Row 4 comes out as `10101` (x=21, f=441) here instead of the noted `10011` —
under a plain single-point crossover at bit 3, pair 2's second child is `10101`, not `10011`. Getting
`10011` for **both** pair-2 offspring would need either a different crossover point, a two-point
crossover, or an extra mutation on that second child that isn't fully legible in the photo. Either
way, the key trend the notes are illustrating still holds: max fitness rose from **729 → 961** and the
average rose too — exactly the GA behaviour described ("if new value is greater than max value, run
another iteration").


## 8. Putting it all together — one reusable GA generation function

In [8]:

def run_one_generation(population, fitness_fn, decode_fn, crossover_points, mutation_positions=None,
                        p_m=0.0, rng=None):
    '''
    Runs ONE generation of a simple binary GA:
      1. Evaluate fitness
      2. Selection via expected-count / stochastic-remainder round-off
      3. Single-point crossover on consecutive pairs
      4. Optional mutation (either random with probability p_m, or forced bit positions
         via mutation_positions = {offspring_index: bit_index} for demonstration purposes)

    Returns: new_population (list[str]), summary DataFrame of the parent generation
    '''
    rng = rng or random.Random()

    xs = [decode_fn(c) for c in population]
    fits = [fitness_fn(x) for x in xs]
    avg_fit = sum(fits) / len(fits)

    expected = [f / avg_fit for f in fits]
    guaranteed = [int(e) for e in expected]
    remainders = [e - g for e, g in zip(expected, guaranteed)]
    slots_left = len(population) - sum(guaranteed)

    order = sorted(range(len(population)), key=lambda i: remainders[i], reverse=True)
    actual = guaranteed[:]
    for i in order[:slots_left]:
        actual[i] += 1

    mating_pool = []
    for chrom, count in zip(population, actual):
        mating_pool.extend([chrom] * count)

    pairs = [(mating_pool[i], mating_pool[i + 1]) for i in range(0, len(mating_pool), 2)]
    offspring = []
    for (p1, p2), cp in zip(pairs, crossover_points):
        c1, c2 = crossover(p1, p2, cp)
        offspring.extend([c1, c2])

    if mutation_positions:
        for idx, bit in mutation_positions.items():
            offspring[idx] = mutate_at(offspring[idx], bit)
    elif p_m > 0:
        offspring = [mutate_random(c, p_m, rng) for c in offspring]

    summary = pd.DataFrame({
        'Chromosome': population, 'x': xs, 'f(x)': fits,
        'Expected count': [round(e, 3) for e in expected],
        'Actual count': actual
    })

    return offspring, summary


# Reproduce the whole worked example in one call
next_gen, summary_table = run_one_generation(
    population=['11011', '10001', '01111', '10111'],
    fitness_fn=fitness,
    decode_fn=decode,
    crossover_points=[3, 3],
    mutation_positions={0: 2, 1: 3}   # replicate the two demonstrated mutations
)

print("Next generation:", next_gen)
summary_table


Next generation: ['11111', '11001', '10011', '10101']


,Chromosome,x,f(x),Expected count,Actual count
0,11011,27,729,1.646,2
1,10001,17,289,0.652,1
2,01111,15,225,0.508,0
3,10111,23,529,1.194,1



## 9. Bonus — Q2 style: real-valued decoding, $x \in [\text{min}, \text{max}]$

Your second question extends the same 5-bit encoding, but decodes to a **real-valued** $x$ in a
custom range $[\text{min}, \text{max}]$ (here $[0, 2]$) instead of the raw integer, using:

$$x = \text{min} + \frac{\text{max} - \text{min}}{2^{L} - 1} \times (\text{decimal value})$$

where $L=5$ is the chromosome length. This is exactly the formula and numbers in your notes
(e.g. `10110` → decimal 22 → $x = 0 + \frac{2}{31}\times 22 = 1.419$).


In [9]:

def decode_real(chromosome: str, x_min: float, x_max: float, length: int = CHROM_LENGTH) -> float:
    d = int(chromosome, 2)
    return x_min + (x_max - x_min) / (2 ** length - 1) * d

def f_minimize(x: float) -> float:
    '''f(x) = -x^2 + 2x  (the function to minimize in Q2).'''
    return -x**2 + 2 * x

q2_population = ['11010', '00111', '10110', '00101']
random_numbers = [0.4, 0.15, 0.7, 0.9]   # as given in the notes

q2_df = pd.DataFrame({'Chromosome': q2_population, 'r': random_numbers})
q2_df['decimal'] = q2_df['Chromosome'].apply(lambda c: int(c, 2))
q2_df['x'] = q2_df['Chromosome'].apply(lambda c: decode_real(c, x_min=0, x_max=2))
q2_df['f(x)'] = q2_df['x'].apply(f_minimize)
q2_df.round(3)


,Chromosome,r,decimal,x,f(x)
0,11010,0.40,26,1.677,0.541
1,00111,0.15,7,0.452,0.699
2,10110,0.70,22,1.419,0.824
3,00101,0.90,5,0.323,0.541



Compare the `x` column above with the notes: `1.677, 0.451, 1.419, 0.322` (yours shows `1.67`,
which rounds from 1.677) — matches ✔.

> The exact role of the random numbers (`0.4, 0.15, 0.7, 0.9`) and the "crossover point is 1 and 5th
> digit" instruction for Q2 were the parts that were hardest for me to read precisely in the photo.
> If you can confirm whether those random numbers are meant for **roulette-wheel selection** or for
> **triggering mutation**, and what "crossover point 1 and 5th digit" means exactly (e.g. a two-point
> crossover keeping bit 1 and bit 5 fixed), I'm happy to extend this section to finish Q2 fully.
